In [1]:
from sedona.spark import *
config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
Setting Spark log level to "WARN".
26/03/12 13:19:59 INFO core/src/lib.rs: Sedona native acceleration engine v0.12.0 ready
                                                                                

In [2]:
database = 'gde_gold'
sedona.sql(f'CREATE DATABASE IF NOT EXISTS org_catalog.{database}')

DataFrame[]

In [3]:
sedona.sql(f'''
create or replace table org_catalog.{database}.analytics_gold as
select
a.sale_id,
a.overture_id,
a.polygon as geometry,
a.height,
a.sale_date,
a.sale_price,
a.city,
a.year_built,
a.beds,
a.bath_3qtr + a.bath_full + a.bath_half as total_bath,
a.sqft,
a.sqft_fbsmt,
a.stories,
b.flood_zone,
c.distance as dist_to_major_intersection,
d.distance as dist_to_park
from org_catalog.gde_bronze.king_co_homes_conflated a
left outer join org_catalog.gde_silver.homes_flood_hazards b using (sale_id)
left outer join org_catalog.gde_silver.roads_proximity c using (sale_id)
left outer join org_catalog.gde_silver.homes_distance_to_seattle d using (sale_id)
''')

DataFrame[]

In [4]:
sedona.sql(f'''
create or replace table org_catalog.{database}.analytics_home_level as
select
geometry,
overture_id,
count(sale_id) as sales_count,
array_agg(sale_price) as price,
array_agg(sale_date) as sale_date
from org_catalog.{database}.analytics_gold
group by 1, 2
order by sale_date asc
''')

DataFrame[]

In [5]:
sedona.sql(f'select * from org_catalog.{database}.analytics_home_level').show()

+--------------------+--------------------+-----------+--------------------+--------------------+
|            geometry|         overture_id|sales_count|               price|           sale_date|
+--------------------+--------------------+-----------+--------------------+--------------------+
|POLYGON ((-122.20...|776329b7-711b-48d...|          1|            [230000]|        [1999-03-04]|
|POLYGON ((-122.20...|29695649-72ec-499...|          1|            [363400]|        [1999-03-29]|
|POLYGON ((-122.20...|b8b120d9-8810-4cd...|          1|            [327000]|        [1999-06-24]|
|POLYGON ((-122.20...|1182f4b5-86a1-4e2...|          3|[277000, 525000, ...|[1999-08-25, 2005...|
|POLYGON ((-122.20...|5ec3beaa-ea82-4f2...|          1|            [331500]|        [2000-01-11]|
|POLYGON ((-122.20...|c21a0a6e-93d5-49d...|          2|   [575000, 1405450]|[2000-02-29, 2019...|
|POLYGON ((-122.21...|edc62360-5881-472...|          1|           [2000000]|        [2000-06-12]|
|POLYGON ((-122.20..

In [6]:
sedona.sql(f'''
create or replace table org_catalog.{database}.ai_ready as
with a as (select * from  org_catalog.gde_bronze.king_co_homes order by sale_date asc)
select
a.city,
EXTRACT(YEAR FROM a.sale_date) AS year,
EXTRACT(MONTH FROM a.sale_date) AS month,
count(a.sale_id) as total_sales,
min(a.sale_price) as min_sale_price,
max(a.sale_price) as max_sale_price,
avg(a.sale_price) as mean_sale_price,
array_agg(a.sale_price) as sale_prices,
array_agg(d.distance) as dist_to_park
from a
left outer join org_catalog.gde_silver.homes_distance_to_seattle d using (sale_id)
group by 1, 2, 3
''')

DataFrame[]

In [7]:
sedona.sql(f'select * from org_catalog.{database}.ai_ready').show()

+------+----+-----+-----------+--------------+--------------+------------------+--------------------+--------------------+
|  city|year|month|total_sales|min_sale_price|max_sale_price|   mean_sale_price|         sale_prices|        dist_to_park|
+------+----+-----+-----------+--------------+--------------+------------------+--------------------+--------------------+
|ALGONA|2001|    1|          1|        162000|        162000|          162000.0|            [162000]| [37723.97374366178]|
|ALGONA|2002|    8|          4|        165000|        190000|          176375.0|[173500, 190000, ...|[37533.3575664516...|
|ALGONA|2004|   12|          4|        216000|        264000|          239975.0|[264000, 216000, ...|[36655.8339530358...|
|ALGONA|2005|    1|          5|        187775|        272000|          223445.0|[227000, 272000, ...|[37120.7282843969...|
|ALGONA|2006|    9|          5|        215000|        255000|          235970.0|[244950, 255000, ...|[36793.7385261016...|
|ALGONA|2009|   

In [8]:
# Confirming table has rows
sedona.sql("SELECT COUNT(*) FROM org_catalog.gde_gold.analytics_gold").show()

+--------+
|count(1)|
+--------+
|     376|
+--------+

